In [ ]:
# Divergence table and estimator stability.
#
# No generation, no training. Runs entirely on data that already exists, so it works
# while gpt-oss-120b is down.
#
# Two outputs:
#   1. corpus vs v14 on every measurement, size-matched  -> the gap table
#   2. how many labelled sentences each measurement needs -> the minimum-n column
#
# Together those are the divergence half of the guide. The mixing curve is the other
# half: what synthetic data is worth at each annotation budget.


# ===========================================================================
# Cell 1: setup. Use the FILTERED human pool.
# The eight enumeration sentences carry 43% of instances and almost no positives, so
# they distort every aggregate: unfiltered positive rate 0.096, filtered 0.162.
# They also cost only 0.010 F1 to remove (0.800 -> 0.790), so filtering is near-free.
# ===========================================================================
import importlib
from collections import Counter

import ddi.divergence
importlib.reload(ddi.divergence)

from ddi.data import build_human
from ddi.manifest import load_dataset
from ddi import divergence as dv

V14_ID = "20260807-123340-ff79db"
LOWROLE_ID = None          # set to the P_ROLE=0.2 dataset id if you want it as an arm

train, dev, val = build_human()
per_sent = Counter(r["sent_id"] for r in train)
monsters = {s for s, n in per_sent.items() if n >= 190}
train_f = [r for r in train if r["sent_id"] not in monsters]

v14, _ = load_dataset(V14_ID)

print(f"corpus (filtered) {len(train_f):>6} inst / "
      f"{len({r['sent_id'] for r in train_f}):>5} sents")
print(f"v14               {len(v14):>6} inst / "
      f"{len({r['sent_id'] for r in v14}):>5} sents")



In [ ]:
import importlib, ddi.divergence
importlib.reload(ddi.divergence)
from ddi import divergence as dv
print([m[0] for m in dv.MEASUREMENTS])

In [ ]:

# ===========================================================================
# Cell 2: the gap table.
# match_size subsamples the larger side to the smaller side's sentence count. The probe
# in particular rises with data volume, so an unmatched comparison flatters whichever
# side has less data.
# ===========================================================================
gap = dv.compare(train_f, v14, match_size=True)
print(gap.to_string(index=False))

# Expected, from measurements already taken by hand:
#   role adjacency NONE   corpus 0.005  v14 0.240   ratio ~48x, the largest divergence
#   role adjacency ratio  corpus 0.27   v14 2.02    inverted direction
#   hard negative rate    corpus ~0.50  v14 0.511   close, this one is fixed
#   count-rule F1         corpus 0.274  v14 0.279   close
#   positive rate         corpus 0.162  v14 0.190
#   probe lift            corpus 0.152  v14 ~0.22   at matched size




In [ ]:

# ===========================================================================
# Cell 3: unfiltered corpus as a second reference.
# Whether the eight monsters belong in the reference is a real question: if standard
# DDI-2013 preprocessing filters them, they should not be in it. Report both.
# ===========================================================================
gap_unf = dv.compare(train, v14, match_size=True)
merged = gap.merge(gap_unf[["measurement", "corpus"]], on="measurement",
                   suffixes=("_filtered", "_unfiltered"))
print(merged[["measurement", "tier", "corpus_filtered", "corpus_unfiltered",
              "synth"]].to_string(index=False))



In [ ]:

# ===========================================================================
# Cell 4: estimator stability. This is the minimum-n column.
# ~10 draws x 6 sizes x 7 measurements. The probe dominates the runtime; if it drags,
# drop max_features or cut draws to 5.
# ===========================================================================
stab = dv.stability_table(train_f, sizes=(50, 100, 200, 500, 1000, 2000), draws=10)

for name, g in stab.groupby("measurement", sort=False):
    print(f"\n{name}   (full corpus = {g.full.iloc[0]})")
    print(f"  {'n':>5} {'median':>8} {'sd':>8} {'bias':>8}  {'sd/full':>8}")
    for _, r in g.iterrows():
        rel = r.sd / abs(r.full) if r.full else float("nan")
        print(f"  {r.n_sentences:>5} {r['median']:>8.4f} {r.sd:>8.4f} "
              f"{r.bias:>8.4f}  {rel:>8.2f}")

# Read: the smallest n where sd is small relative to the corpus value AND bias is near
# zero. Some measurements will be usable at 100 sentences, some not at 2000. That
# asymmetry is the finding: composition is cheap to estimate, prose distribution is not.



In [ ]:
# Cell 5: minimum annotation budget, on a DETECTION criterion.
#
# The earlier version asked "how many labelled sentences before this measurement is
# estimated precisely", using sd/full. That is the wrong question and it misjudged the
# role adjacency gap badly: corpus value -0.0093, so sd/full stays above 1.0 until
# n=200 and only reaches 0.11 at 2000, which reads as "needs the whole corpus". But the
# divergence to detect is +0.245, and sd at n=50 is 0.016. Fifteen sigma. Detectable
# with fifty sentences.
#
# The right question is "how many labelled sentences before I can tell this dataset
# apart from the corpus on this measurement". That depends on the effect size, so the
# minimum-n column is per comparison, not per measurement. A measurement that needs
# 2000 sentences to pin down to three decimal places may need 50 to catch a gross
# divergence.
#
# Criterion: smallest n where |corpus - synth| > 3 * sd(corpus estimate at n).
# Three sigma is arbitrary but stated, which is the improvement over the invented
# thresholds in gates.py.
#
# Where the gap is already near zero the criterion returns nothing, and that is correct:
# there is no divergence to detect. Reported as "matched" rather than as a failure.

import pandas as pd
import numpy as np

SIGMA = 3.0

rows = []
for name, g in stab.groupby("measurement", sort=False):
    g = g.sort_values("n_sentences")
    corpus = g.full.iloc[0]
    synth = float(gap.loc[gap.measurement == name, "synth"].iloc[0])
    effect = abs(synth - corpus)

    detect = g[g.sd * SIGMA < effect]
    min_n = int(detect.n_sentences.min()) if len(detect) else None

    # separately: n needed to estimate the corpus value itself, for reference
    est = g[(g.sd < 0.2 * abs(corpus)) & (g.bias.abs() < 0.1 * abs(corpus))] \
        if corpus else g.iloc[0:0]
    est_n = int(est.n_sentences.min()) if len(est) else None

    rows.append({
        "measurement": name,
        "tier": g.tier.iloc[0],
        "corpus": round(corpus, 4),
        "synth": round(synth, 4),
        "effect": round(synth - corpus, 4),
        "sd@50": round(g[g.n_sentences == 50].sd.iloc[0], 4),
        "detect_n": min_n,
        "estimate_n": est_n,
    })

table = pd.DataFrame(rows)


def fmt(v, matched="matched"):
    return matched if v is None else str(v)


print(f"detection at {SIGMA:.0f} sigma. 'matched' = no divergence to detect.\n")
print(f"{'measurement':<24} {'tier':<15} {'corpus':>8} {'synth':>8} "
      f"{'effect':>8} {'detect':>8} {'estimate':>9}")
for _, r in table.iterrows():
    print(f"{r.measurement:<24} {r.tier:<15} {r.corpus:>8.4f} {r.synth:>8.4f} "
          f"{r.effect:>8.4f} {fmt(r.detect_n):>8} {fmt(r.estimate_n, 'n/a'):>9}")

print("""
Read:
  detect_n    labelled sentences needed to see THIS divergence
  estimate_n  labelled sentences needed to pin the corpus value down for its own sake

The two columns come apart, and that is the point. Detectability is driven by effect
size; estimation is driven by variance alone. A practitioner checking a new synthetic
set cares about the first.

'matched' rows are where v14 already agrees with the corpus. Those are the composition
properties the build gates were written for, and they are fixed, which is why the
remaining divergences all sit in things the gates never measured.
""")

# unlabelled measurements need no annotation at all; report them separately so the
# labelled table is not padded with zeros
unl = gap[gap.tier == "unlabelled"]
print("unlabelled tier, no annotation required:")
for _, r in unl.iterrows():
    print(f"  {r.measurement:<24} corpus {r.corpus:>8.4f}  synth {r.synth:>8.4f}  "
          f"gap {r.gap:>+8.4f}")

table.to_csv("figures/divergence-minimum-n.csv", index=False)